In [ ]:
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


## Loading data from Huggingface

In [ ]:
ds = load_dataset("maxstiles/final_dataset")

In [ ]:
print(ds)
print(ds.keys())

split_name = list(ds.keys())[0]
print("Using split:", split_name)

print(ds[split_name][0])
print(ds[split_name].features)

In [ ]:
class GPSImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None, lat_mean=None, lat_std=None, lon_mean=None, lon_std=None):
        self.hf_dataset = hf_dataset
        self.transform = transform

        # Compute mean and std from the dataframe if not provided
        self.latitude_mean = lat_mean if lat_mean is not None else np.mean(np.array(self.hf_dataset['latitude']))
        self.latitude_std = lat_std if lat_std is not None else np.std(np.array(self.hf_dataset['latitude']))
        self.longitude_mean = lon_mean if lon_mean is not None else np.mean(np.array(self.hf_dataset['longitude']))
        self.longitude_std = lon_std if lon_std is not None else np.std(np.array(self.hf_dataset['longitude']))

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        # Extract data
        example = self.hf_dataset[idx]

        # Load and process the image
        image = example['image']
        latitude = example['latitude']
        longitude = example['longitude']
        # image = image.rotate(-90, expand=True)
        if self.transform:
            image = self.transform(image)

        # Normalize GPS coordinates
        latitude = (latitude - self.latitude_mean) / self.latitude_std
        longitude = (longitude - self.longitude_mean) / self.longitude_std
        gps_coords = torch.tensor([latitude, longitude], dtype=torch.float32)

        return image, gps_coords

## Experiment Setup

This notebook now runs a lightweight **ResNet-18** augmentation sweep so we can compare
resize, flip, rotation, and color-jitter settings before building the leaderboard
submission.


In [ ]:
SEED = 42
DEFAULT_BATCH_SIZE = 32
SWEEP_EPOCHS = 3
FINALIST_EPOCHS = 5
TOP_K_FINALISTS = 2
DROPOUT = 0.2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0 if "google.colab" in sys.modules else min(2, os.cpu_count() or 1)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

CHECKPOINT_DIR = Path("augmentation_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_train_transform(image_size=224, hflip_p=0.0, rotation_deg=0, color_jitter=None):
    ops = [transforms.Resize((image_size, image_size))]

    if hflip_p > 0:
        ops.append(transforms.RandomHorizontalFlip(p=hflip_p))

    if rotation_deg > 0:
        ops.append(transforms.RandomRotation(degrees=rotation_deg))

    if color_jitter is not None:
        ops.append(transforms.ColorJitter(**color_jitter))

    ops.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )
    return transforms.Compose(ops)


def build_eval_transform(image_size=224):
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


EXPERIMENT_CONFIGS = [
    {
        "name": "resnet18_resize224_baseline",
        "image_size": 224,
        "hflip_p": 0.0,
        "rotation_deg": 0,
        "color_jitter": None,
    },
    {
        "name": "resnet18_resize224_flip",
        "image_size": 224,
        "hflip_p": 0.5,
        "rotation_deg": 0,
        "color_jitter": None,
    },
    {
        "name": "resnet18_resize224_rotate10",
        "image_size": 224,
        "hflip_p": 0.0,
        "rotation_deg": 10,
        "color_jitter": None,
    },
    {
        "name": "resnet18_resize224_jitter_light",
        "image_size": 224,
        "hflip_p": 0.0,
        "rotation_deg": 0,
        "color_jitter": {
            "brightness": 0.1,
            "contrast": 0.1,
            "saturation": 0.1,
            "hue": 0.02,
        },
    },
    {
        "name": "resnet18_resize224_combo_light",
        "image_size": 224,
        "hflip_p": 0.5,
        "rotation_deg": 10,
        "color_jitter": {
            "brightness": 0.1,
            "contrast": 0.1,
            "saturation": 0.1,
            "hue": 0.02,
        },
    },
    {
        "name": "resnet18_resize256_combo_light",
        "image_size": 256,
        "hflip_p": 0.5,
        "rotation_deg": 10,
        "color_jitter": {
            "brightness": 0.1,
            "contrast": 0.1,
            "saturation": 0.1,
            "hue": 0.02,
        },
    },
    {
        "name": "resnet18_resize256_combo_medium",
        "image_size": 256,
        "hflip_p": 0.5,
        "rotation_deg": 15,
        "color_jitter": {
            "brightness": 0.2,
            "contrast": 0.2,
            "saturation": 0.2,
            "hue": 0.03,
        },
    },
]

set_seed(SEED)

pd.DataFrame(EXPERIMENT_CONFIGS)


In [ ]:
USE_DRIVE_CHECKPOINTS = True

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE_CHECKPOINTS:
    drive.mount("/content/drive", force_remount=False)
    DRIVE_RUN_DIR = Path("/content/drive/MyDrive/CIS5190_runs/test_transform")
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

    CHECKPOINT_DIR = DRIVE_RUN_DIR / "augmentation_checkpoints"
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    print("Saving checkpoints to Google Drive:", CHECKPOINT_DIR)
else:
    print("Saving checkpoints locally:", CHECKPOINT_DIR.resolve())

print("NUM_WORKERS =", NUM_WORKERS)


In [ ]:
def create_datasets_and_loaders(config, batch_size=DEFAULT_BATCH_SIZE):
    train_transform = build_train_transform(
        image_size=config["image_size"],
        hflip_p=config["hflip_p"],
        rotation_deg=config["rotation_deg"],
        color_jitter=config["color_jitter"],
    )
    eval_transform = build_eval_transform(image_size=config["image_size"])

    train_dataset = GPSImageDataset(hf_dataset=ds["train"], transform=train_transform)

    val_dataset = GPSImageDataset(
        hf_dataset=ds["validation"],
        transform=eval_transform,
        lat_mean=train_dataset.latitude_mean,
        lat_std=train_dataset.latitude_std,
        lon_mean=train_dataset.longitude_mean,
        lon_std=train_dataset.longitude_std,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return train_dataset, val_dataset, train_loader, val_loader


preview_config = EXPERIMENT_CONFIGS[4]
train_dataset, val_dataset, train_dataloader, val_dataloader = create_datasets_and_loaders(preview_config)

print("Preview config:", preview_config["name"])
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Image size:", preview_config["image_size"])


In [ ]:
for images, gps_coords in train_dataloader:
    print("Training batch image shape:", images.size())
    print("Training batch target shape:", gps_coords.size())
    break


In [ ]:
print("Latitude mean:", train_dataset.latitude_mean)
print("Latitude std:", train_dataset.latitude_std)
print("Longitude mean:", train_dataset.longitude_mean)
print("Longitude std:", train_dataset.longitude_std)


## Data Examples

In [ ]:
def denormalize(tensor, mean, std):
    mean = np.array(mean)
    std = np.array(std)
    tensor = tensor.numpy().transpose((1, 2, 0))
    tensor = std * tensor + mean
    tensor = np.clip(tensor, 0, 1)
    return tensor


data_iter = iter(train_dataloader)
images, gps_coords = next(data_iter)

for idx, image_tensor in enumerate(images[:4]):
    image = denormalize(image_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)

    latitude = gps_coords[idx][0].item() * train_dataset.latitude_std + train_dataset.latitude_mean
    longitude = gps_coords[idx][1].item() * train_dataset.longitude_std + train_dataset.longitude_mean

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Lat: {latitude:.5f}, Lon: {longitude:.5f}")
    plt.axis("off")
    plt.show()


## Model

`torchvision` does not ship a standard ResNet-16, so this sweep uses **ResNet-18**
as the lightweight baseline.


In [ ]:
class Image2GPSModel(nn.Module):
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()

        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet18(weights=weights)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.backbone(x)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Image2GPSModel(pretrained=True, dropout=DROPOUT).to(device)
print("Device:", device)


## Train Sweep

The sweep keeps the model and optimizer fixed, and only changes augmentation parameters.
Experiments are ranked by **validation mean Haversine distance** because that matches the
leaderboard metric.


In [ ]:
criterion = nn.SmoothL1Loss(beta=0.5)

print(f"Sweep epochs per config: {SWEEP_EPOCHS}")
print(f"Top finalists to rerun later: {TOP_K_FINALISTS}")
print(f"Checkpoint directory: {CHECKPOINT_DIR.resolve()}")


In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Compute Haversine distance in meters.
    Inputs should be numpy arrays in degrees.
    """
    R = 6371000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


def evaluate_model(
    model,
    data_loader,
    device,
    lat_mean,
    lat_std,
    lon_mean,
    lon_std,
    criterion=None,
):
    model.eval()

    total_loss = 0.0
    all_pred_lat = []
    all_pred_lon = []
    all_true_lat = []
    all_true_lon = []

    with torch.no_grad():
        for images, targets in data_loader:
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)

            if criterion is not None:
                total_loss += criterion(preds, targets).item() * images.size(0)

            preds = preds.cpu().numpy()
            targets = targets.cpu().numpy()

            pred_lat = preds[:, 0] * lat_std + lat_mean
            pred_lon = preds[:, 1] * lon_std + lon_mean

            true_lat = targets[:, 0] * lat_std + lat_mean
            true_lon = targets[:, 1] * lon_std + lon_mean

            all_pred_lat.extend(pred_lat)
            all_pred_lon.extend(pred_lon)
            all_true_lat.extend(true_lat)
            all_true_lon.extend(true_lon)

    all_pred_lat = np.array(all_pred_lat)
    all_pred_lon = np.array(all_pred_lon)
    all_true_lat = np.array(all_true_lat)
    all_true_lon = np.array(all_true_lon)

    distances = haversine_distance(all_true_lat, all_true_lon, all_pred_lat, all_pred_lon)
    mean_distance = float(distances.mean())
    median_distance = float(np.median(distances))
    val_loss = total_loss / len(data_loader.dataset) if criterion is not None else None

    metrics = {
        "val_loss": val_loss,
        "mean_haversine_m": mean_distance,
        "median_haversine_m": median_distance,
        "min_haversine_m": float(distances.min()),
        "max_haversine_m": float(distances.max()),
    }

    return metrics, (all_pred_lat, all_pred_lon, all_true_lat, all_true_lon)


In [ ]:
def train_one_experiment(config, num_epochs=SWEEP_EPOCHS, pretrained=True):
    set_seed(SEED)

    train_dataset, val_dataset, train_loader, val_loader = create_datasets_and_loaders(config)

    model = Image2GPSModel(pretrained=pretrained, dropout=DROPOUT).to(device)
    criterion = nn.SmoothL1Loss(beta=0.5)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    checkpoint_path = CHECKPOINT_DIR / f"{config['name']}.pt"
    history = []
    best_summary = None
    best_distance = float("inf")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for images, gps_coords in tqdm(
            train_loader,
            desc=f"{config['name']} | Epoch {epoch + 1}/{num_epochs}",
            leave=False,
        ):
            images = images.to(device)
            gps_coords = gps_coords.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, gps_coords)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        metrics, _ = evaluate_model(
            model=model,
            data_loader=val_loader,
            device=device,
            lat_mean=train_dataset.latitude_mean,
            lat_std=train_dataset.latitude_std,
            lon_mean=train_dataset.longitude_mean,
            lon_std=train_dataset.longitude_std,
            criterion=criterion,
        )

        record = {
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": metrics["val_loss"],
            "mean_haversine_m": metrics["mean_haversine_m"],
            "median_haversine_m": metrics["median_haversine_m"],
        }
        history.append(record)

        print(
            f"{config['name']} | "
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {metrics['val_loss']:.4f} | "
            f"Mean Haversine: {metrics['mean_haversine_m']:.2f} m"
        )

        if metrics["mean_haversine_m"] < best_distance:
            best_distance = metrics["mean_haversine_m"]
            best_summary = {
                "name": config["name"],
                "image_size": config["image_size"],
                "hflip_p": config["hflip_p"],
                "rotation_deg": config["rotation_deg"],
                "color_jitter": config["color_jitter"],
                "best_epoch": epoch + 1,
                "best_train_loss": float(train_loss),
                "best_val_loss": metrics["val_loss"],
                "best_val_mean_haversine_m": metrics["mean_haversine_m"],
                "best_val_median_haversine_m": metrics["median_haversine_m"],
                "checkpoint_path": str(checkpoint_path),
            }

            checkpoint = {
                "epoch": epoch + 1,
                "config": config,
                "history": history,
                "best_distance": best_distance,
                "best_summary": best_summary,
                "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state_dict": optimizer.state_dict(),
            }
            torch.save(checkpoint, checkpoint_path)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "config": config,
        "history": history,
        "best_summary": best_summary,
        "checkpoint_path": str(checkpoint_path),
    }


In [ ]:
def resume_experiment_from_checkpoint(checkpoint_path, extra_epochs=1, pretrained=True):
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if "model_state_dict" not in checkpoint:
        raise ValueError(
            "This checkpoint only contains model weights. "
            "Use the updated notebook to save full resumable checkpoints."
        )

    config = checkpoint["config"]
    train_dataset, val_dataset, train_loader, val_loader = create_datasets_and_loaders(config)

    model = Image2GPSModel(pretrained=pretrained, dropout=DROPOUT).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])

    criterion = nn.SmoothL1Loss(beta=0.5)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    history = checkpoint.get("history", [])
    best_distance = checkpoint.get("best_distance", float("inf"))
    best_summary = checkpoint.get("best_summary")
    start_epoch = checkpoint.get("epoch", len(history))

    for epoch in range(start_epoch, start_epoch + extra_epochs):
        model.train()
        train_loss = 0.0

        for images, gps_coords in tqdm(
            train_loader,
            desc=f"resume {config['name']} | Epoch {epoch + 1}/{start_epoch + extra_epochs}",
            leave=False,
        ):
            images = images.to(device)
            gps_coords = gps_coords.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, gps_coords)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        metrics, _ = evaluate_model(
            model=model,
            data_loader=val_loader,
            device=device,
            lat_mean=train_dataset.latitude_mean,
            lat_std=train_dataset.latitude_std,
            lon_mean=train_dataset.longitude_mean,
            lon_std=train_dataset.longitude_std,
            criterion=criterion,
        )

        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": float(train_loss),
                "val_loss": metrics["val_loss"],
                "mean_haversine_m": metrics["mean_haversine_m"],
                "median_haversine_m": metrics["median_haversine_m"],
            }
        )

        print(
            f"resume {config['name']} | "
            f"Epoch {epoch + 1} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {metrics['val_loss']:.4f} | "
            f"Mean Haversine: {metrics['mean_haversine_m']:.2f} m"
        )

        if metrics["mean_haversine_m"] < best_distance:
            best_distance = metrics["mean_haversine_m"]
            best_summary = {
                "name": config["name"],
                "image_size": config["image_size"],
                "hflip_p": config["hflip_p"],
                "rotation_deg": config["rotation_deg"],
                "color_jitter": config["color_jitter"],
                "best_epoch": epoch + 1,
                "best_train_loss": float(train_loss),
                "best_val_loss": metrics["val_loss"],
                "best_val_mean_haversine_m": metrics["mean_haversine_m"],
                "best_val_median_haversine_m": metrics["median_haversine_m"],
                "checkpoint_path": str(checkpoint_path),
            }

            updated_checkpoint = {
                "epoch": epoch + 1,
                "config": config,
                "history": history,
                "best_distance": best_distance,
                "best_summary": best_summary,
                "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state_dict": optimizer.state_dict(),
            }
            torch.save(updated_checkpoint, checkpoint_path)

    return {
        "config": config,
        "history": history,
        "best_summary": best_summary,
        "checkpoint_path": str(checkpoint_path),
    }


# Example:
# resume_run = resume_experiment_from_checkpoint(
#     "/content/drive/MyDrive/CIS5190_runs/test_transform/augmentation_checkpoints/resnet18_resize224_combo_light.pt",
#     extra_epochs=2,
# )


In [ ]:
experiment_runs = []

for config in EXPERIMENT_CONFIGS:
    print("\n" + "=" * 80)
    print(f"Running experiment: {config['name']}")
    run_result = train_one_experiment(config, num_epochs=SWEEP_EPOCHS)
    experiment_runs.append(run_result)

sweep_results_df = (
    pd.DataFrame([run["best_summary"] for run in experiment_runs])
    .sort_values("best_val_mean_haversine_m")
    .reset_index(drop=True)
)

sweep_results_df


In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(
    sweep_results_df["name"],
    sweep_results_df["best_val_mean_haversine_m"],
    color="steelblue",
)
plt.xlabel("Best validation mean Haversine distance (meters)")
plt.ylabel("Experiment")
plt.title("Augmentation sweep results")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.show()

best_experiment_name = sweep_results_df.iloc[0]["name"]
finalist_names = sweep_results_df.head(TOP_K_FINALISTS)["name"].tolist()

best_experiment = next(
    run for run in experiment_runs if run["best_summary"]["name"] == best_experiment_name
)
best_history_df = pd.DataFrame(best_experiment["history"])

print("Best config after the short sweep:", best_experiment_name)
print("Top configs to rerun longer:", finalist_names)

best_history_df


## Best Experiment Inspection

The cells below reload the best short-sweep checkpoint, evaluate it on the validation
set, and visualize how far predictions land from the true GPS coordinates.


In [ ]:
best_config = next(config for config in EXPERIMENT_CONFIGS if config["name"] == best_experiment_name)
best_train_dataset, best_val_dataset, best_train_loader, best_val_loader = create_datasets_and_loaders(best_config)

best_model = Image2GPSModel(pretrained=True, dropout=DROPOUT).to(device)
loaded_checkpoint = torch.load(best_experiment["checkpoint_path"], map_location=device)

if "model_state_dict" in loaded_checkpoint:
    best_model.load_state_dict(loaded_checkpoint["model_state_dict"])
else:
    best_model.load_state_dict(loaded_checkpoint)

best_metrics, _ = evaluate_model(
    model=best_model,
    data_loader=best_val_loader,
    device=device,
    lat_mean=best_train_dataset.latitude_mean,
    lat_std=best_train_dataset.latitude_std,
    lon_mean=best_train_dataset.longitude_mean,
    lon_std=best_train_dataset.longitude_std,
    criterion=criterion,
)

best_metrics


In [ ]:
def collect_predictions(model, data_loader, dataset, device):
    model.eval()

    all_true_lat = []
    all_true_lon = []
    all_pred_lat = []
    all_pred_lon = []

    lat_mean = dataset.latitude_mean
    lat_std = dataset.latitude_std
    lon_mean = dataset.longitude_mean
    lon_std = dataset.longitude_std

    with torch.no_grad():
        for images, targets in data_loader:
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)

            preds = preds.cpu().numpy()
            targets = targets.cpu().numpy()

            pred_lat = preds[:, 0] * lat_std + lat_mean
            pred_lon = preds[:, 1] * lon_std + lon_mean

            true_lat = targets[:, 0] * lat_std + lat_mean
            true_lon = targets[:, 1] * lon_std + lon_mean

            all_pred_lat.extend(pred_lat)
            all_pred_lon.extend(pred_lon)
            all_true_lat.extend(true_lat)
            all_true_lon.extend(true_lon)

    results_df = pd.DataFrame(
        {
            "true_lat": all_true_lat,
            "true_lon": all_true_lon,
            "pred_lat": all_pred_lat,
            "pred_lon": all_pred_lon,
        }
    )

    results_df["haversine_m"] = haversine_distance(
        results_df["true_lat"].to_numpy(),
        results_df["true_lon"].to_numpy(),
        results_df["pred_lat"].to_numpy(),
        results_df["pred_lon"].to_numpy(),
    )

    return results_df


In [ ]:
results_df = collect_predictions(
    model=best_model,
    data_loader=best_val_loader,
    dataset=best_val_dataset,
    device=device,
)

results_df.sort_values("haversine_m").head()


In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    results_df["true_lon"],
    results_df["true_lat"],
    label="Actual GPS",
    alpha=0.8,
)
plt.scatter(
    results_df["pred_lon"],
    results_df["pred_lat"],
    label="Predicted GPS",
    alpha=0.8,
)

for _, row in results_df.iterrows():
    plt.plot(
        [row["true_lon"], row["pred_lon"]],
        [row["true_lat"], row["pred_lat"]],
        alpha=0.25,
        linewidth=0.8,
    )

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Actual vs predicted GPS ({best_experiment_name})")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.show()
